In [1]:
import polars as pl
from pathlib import Path
from common.repositories import ContractDescriptionsRepository, EquityBarRepository

In [2]:
root = Path("/Users/mohamedali/trading_project/earnings_ibapi")

In [3]:
repo = ContractDescriptionsRepository(root / "data")

In [4]:
repo.base

PosixPath('/Users/mohamedali/trading_project/earnings_ibapi/data')

In [5]:
res = repo.load_contracts_by_symbol(symbol="BABA")

In [6]:
type(res)

pyarrow.lib.Table

In [7]:
res.to_pylist()

[{'symbol': 'BABA',
  'sec_type': 'STK',
  'conid': 166090175,
  'primary_exchange': 'NYSE',
  'currency': 'USD',
  'sec_id_type': None,
  'sec_id': None,
  'derivative_sec_types': ['CFD', 'OPT', 'IOPT', 'WAR', 'BAG'],
  'created_at': None}]

In [8]:
from common.repositories import ContractRepository

contracts_repo = ContractRepository(root / "data")

In [9]:
contracts_repo.present_conids()

{166090175}

In [10]:
payload = {
    'bars': [
        {'date': '20250825', 'open': 125.18, 'high': 126.73, 'low': 123.5, 'close': 124.35, 'volume': '8060369', 'wap': '125.139', 'barCount': 42855},
        {'date': '20250826', 'open': 125.3, 'high': 126.0, 'low': 123.72, 'close': 124.19, 'volume': '5708497', 'wap': '124.647', 'barCount': 32245},
        {'date': '20250827', 'open': 121.2, 'high': 122.52, 'low': 120.79, 'close': 122.23, 'volume': '8228326', 'wap': '121.708', 'barCount': 47536},
        {'date': '20250828', 'open': 119.53, 'high': 121.25, 'low': 117.6, 'close': 119.57, 'volume': '9952646', 'wap': '119.866', 'barCount': 54776},
        {'date': '20250829', 'open': 128.88, 'high': 136.65, 'low': 128.51, 'close': 135.0, 'volume': '59052476', 'wap': '133.087', 'barCount': 269197}
    ],
    'contract': {'conId': 166090175, 'symbol': 'BABA', 'secType': 'STK', 'currency': 'USD', 'exchange': 'SMART'},
    'parameters': {'end_date_time': '20250830 23:59:59', 'duration_str': '5 D', 'bar_size_setting': '1 day', 'what_to_show': 'TRADES'}
}

In [11]:
bars_repo = EquityBarRepository(root / "data")

In [12]:
import pandas as pd

bars = payload.get("bars", [])
contract = payload.get("contract", {})
parameters = payload.get("parameters", {})
symbol = contract.get("symbol", "N/A")

try:
    df = pd.DataFrame(bars)
    df["time"] = pd.to_datetime(df["date"])
    df["date"] = df["time"].dt.date
    df["symbol"] = contract.get('symbol', None)
    df["bar_size"] = parameters.get("bar_size_setting", None)
    df["data_type"] = parameters.get("what_to_show", None)
    df["created_at"] = pd.Timestamp.now()


    bars_repo.save(df)
except Exception as e:
    print(f"Error saving bars for {symbol}: {e}")

Error saving bars for BABA: name 'pq' is not defined


In [18]:
from pathlib import Path
import polars as pl

base = Path(root) / "data/contract_descriptions"
files = sorted(p for p in base.rglob("*.parquet") if p.is_file())
if not files:
    raise FileNotFoundError(f"No .parquet files under {base}")

In [19]:
base

PosixPath('/Users/mohamedali/trading_project/earnings_ibapi/data/contract_descriptions')

In [21]:
src = f"{root}/data/contract_descriptions/*/*/*.parquet"
lf = pl.scan_parquet(src)

In [22]:
lf.head(5).collect()

conid,symbol,sec_type,currency,primary_exchange,derivative_sec_types
i64,str,str,str,str,list[str]
-1,"""""","""BOND""","""""","""""",[]
8314,"""IBM""","""STK""","""USD""","""NYSE""","[""CFD"", ""OPT"", … ""BAG""]"
1411277,"""IBM""","""STK""","""EUR""","""IBIS""","[""IOPT""]"
38709473,"""IBM""","""STK""","""MXN""","""MEXI""",[]
530091934,"""IBM""","""STK""","""CAD""","""AEQLIT""",[]


In [24]:
lf.filter(pl.col("derivative_sec_types").list.contains("OPT")).collect()

conid,symbol,sec_type,currency,primary_exchange,derivative_sec_types
i64,str,str,str,str,list[str]
8314,"""IBM""","""STK""","""USD""","""NYSE""","[""CFD"", ""OPT"", … ""BAG""]"


In [13]:
import pyarrow as pa
import pyarrow.compute as pc

In [16]:
values = pa.list_(pa.string())

In [20]:
s = pa.array(["s1", "s2"], type=pa.string())
v = pa.array([["v1", "v2"], []], type=pa.list_(pa.string()))

In [23]:
t = pa.table([s, v], names=["s", "v"])

In [24]:
col = t.column("v")

In [27]:
pc.is_in(col, [])

TypeError: "[]" is not a valid value set

In [28]:
flattened = pc.list_flatten(t["v"])

In [29]:
flattened

[
  [
    "v1",
    "v2"
  ]
]

In [31]:
mask_flat = pc.equal(flattened, pa.scalar("v1"))

In [32]:
mask_flat

[
  [
    true,
    false
  ]
]

In [33]:
mask_parent = pc.list_parent_indices(t["v"]).take(pc.indices_nonzero(mask_flat)).unique()


In [34]:
mask_parent

[
  0
]

In [35]:
rows_with_v1 = pc.is_in(pa.array(range(len(t))), mask_parent)

In [36]:
rows_with_v1

[
  true,
  false
]

In [37]:
filtered = t.filter(rows_with_v1)

In [38]:
filtered

pyarrow.Table
s: string
v: list<item: string>
  child 0, item: string
----
s: [["s1"]]
v: [[["v1","v2"]]]

In [39]:
print(filtered.to_pandas())

    s         v
0  s1  [v1, v2]


In [40]:
import pyarrow.compute as pc

mask = pc.list_contains(t["v"], "v1")
filtered = t.filter(mask)

AttributeError: module 'pyarrow.compute' has no attribute 'list_contains'